# LAB01｜Google ADK + Groq：第一個真正 Agent
## V2.1 學員教學版｜Google ADK + Groq + Agent Skills + MCP + LINE

> **適合對象：** 不熟 Python、第一次接觸 AI Agent 的學員  
> **操作方式：** 由上到下依序按 ▶ 執行。  
> **重要原則：** 看到「👨‍🎓 學生修改區」才需要動手，其餘程式請先不要修改。


## 🎯 LAB01 學習目標

完成後你應該能回答：

1. Agent 和一般聊天模型差在哪裡？
2. Tool 是什麼？
3. Google ADK 負責什麼？
4. Groq API 在整個架構中扮演什麼角色？

### 本 LAB 架構

```text
使用者
   ↓
Google ADK Agent
   ↓
Groq 模型
   ↓
Agent 自己選 Tool
   ↓
查產品 / 查成本
```


In [ ]:
# ============================================================
# STEP 1｜安裝本課需要的套件
# ============================================================
# 這一格的目的：
#   把 Google ADK、MCP、LiteLLM、pandas 安裝到 Colab。
#
# 為什麼需要？
#   Colab 預設沒有這些套件，因此第一次執行一定要安裝。
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。直接按左側 ▶ 執行。
#
# ✅ 成功畫面：
#   執行結束後沒有紅色錯誤訊息即可。
#
# ❗常見錯誤：
#   1. 安裝很久：正常，第一次需要下載套件。
#   2. 出現版本衝突：請重新啟動執行階段後再執行一次。
# ============================================================

!pip -q install "google-adk[mcp]>=2.0.0,<3.0.0" "litellm>=1.84,<2" pandas
print("✅ 套件安裝完成")


In [ ]:
# ============================================================
# STEP 2｜上傳課程資料包
# ============================================================
# 這一格的目的：
#   把「企業模擬資料、Agent Skill、MCP Server」一次載入 Colab。
#
# 你要準備的檔案：
#   quotation_agent_v2_runtime.zip
#
# 👨‍🎓 學生修改區：
#   不用改程式，只要執行後選擇上述 ZIP 檔。
#
# ✅ 成功畫面：
#   看到「✅ 課程資料已載入」
#
# ❗常見錯誤：
#   - 上傳錯檔：重新執行這一格並選正確 ZIP。
#   - 沒看到 skills/ 或 data/：代表 ZIP 沒有成功解壓。
# ============================================================

from google.colab import files
import zipfile, os

if not os.path.exists("skills/quotation-decision/SKILL.md"):
    uploaded = files.upload()
    package = next((n for n in uploaded if n.endswith(".zip")), None)
    if not package:
        raise RuntimeError("❌ 請上傳 quotation_agent_v2_runtime.zip")
    with zipfile.ZipFile(package, "r") as z:
        z.extractall(".")

print("✅ 課程資料已載入")
print("目前重要資料夾：")
print(" - data/")
print(" - skills/quotation-decision/")
print(" - mcp_server/")


In [ ]:
# ============================================================
# STEP 3｜從 Colab Secrets 讀取 Groq API Key
# ============================================================
# 這一格的目的：
#   從 Google Colab 左側的「🔑 Secrets」安全讀取 GROQ_API_KEY。
#
# 👨‍🎓 學生事前準備：
#   1. 打開 Colab 左側選單
#   2. 點「🔑 Secrets」
#   3. 新增名稱：GROQ_API_KEY
#   4. Value 貼上你的 Groq API Key
#   5. 開啟「Notebook access」授權
#
# ⚠️ 不要把 API Key 直接寫在 Notebook 程式碼中。
#
# ✅ 成功畫面：
#   顯示「✅ 已從 Colab Secrets 讀取 GROQ_API_KEY」
#
# ❗常見錯誤：
#   KeyError / Secret not found
#   → 代表 Secrets 裡沒有 GROQ_API_KEY，或沒有開 Notebook access。
# ============================================================

import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY 為空值")

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    # 課堂穩定模型
    GROQ_MODEL = "groq/qwen/qwen3.8-27b"

    print("✅ 已從 Colab Secrets 讀取 GROQ_API_KEY")
    print("✅ Groq 設定完成")
    print("Model =", GROQ_MODEL)
    print("教學穩定模式：reasoning_effort = none")

except Exception as e:
    raise RuntimeError(
        "❌ 無法讀取 Colab Secret：GROQ_API_KEY\n\n"
        "請到 Colab 左側「🔑 Secrets」新增：\n"
        "名稱：GROQ_API_KEY\n"
        "值：你的 Groq API Key\n"
        "並開啟 Notebook access。\n\n"
        f"原始錯誤：{e}"
    )


## STEP 4｜讀取企業資料

這一格不是 Agent。

我們只是先把模擬企業資料讀進 Colab，等一下再讓 Agent 使用。


In [ ]:
# ============================================================
# STEP 4｜讀取企業資料
# ============================================================
# pandas 可以把 CSV 讀成表格。
#
# PRODUCTS = 產品主檔
# COSTS    = 成本主檔
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

import pandas as pd

PRODUCTS = pd.read_csv("data/products.csv")
COSTS = pd.read_csv("data/costs.csv")

print("✅ 企業資料讀取完成")
print("產品筆數：", len(PRODUCTS))
print("成本筆數：", len(COSTS))

display(PRODUCTS.head())


## STEP 5｜建立兩個 Tools

Tool 就像 Agent 可以使用的「公司功能」。

這裡先提供兩個能力：

- `product_lookup()`：查產品規格
- `latest_cost()`：查最新成本

**重點：Tool 本身不是 AI。它只是可靠地取得資料。**


In [ ]:
# ============================================================
# STEP 5｜建立 Tool
# ============================================================
#
# Tool 1：product_lookup
# 輸入產品代號，例如 CV420H
# 回傳產品規格、耐熱、RoHS、MOQ、交期等資料
#
# Tool 2：latest_cost
# 輸入產品代號
# 回傳最新單位成本
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。先觀察「普通 Python function 如何變成 Agent Tool」。
# ============================================================

def product_lookup(product_code: str) -> dict:
    """查詢產品規格、耐熱、RoHS、MOQ 與標準交期。"""
    row = PRODUCTS[
        PRODUCTS.product_code.str.upper() == product_code.upper()
    ]
    return {"found": False} if row.empty else {
        "found": True,
        **row.iloc[0].to_dict()
    }

def latest_cost(product_code: str) -> dict:
    """查詢產品最新單位成本。只有需要成本資料時才使用。"""
    row = COSTS[
        COSTS.product_code.str.upper() == product_code.upper()
    ].sort_values("effective_month")

    return {"found": False} if row.empty else {
        "found": True,
        **row.iloc[-1].to_dict()
    }

print("✅ Tools 建立完成")
print("測試 product_lookup：")
print(product_lookup("CV420H"))


## STEP 6｜建立第一個 Google ADK Agent

一個最基本的 Agent 可以先記住三件事：

```text
Agent
├─ Model       → 大腦
├─ Instruction → 工作規則
└─ Tools       → 可以使用的能力
```


In [ ]:
# ============================================================
# STEP 6｜建立第一個 AI Agent
# ============================================================
#
# Model：
#   使用 Groq API 上的 GPT-OSS 模型
#
# Instruction：
#   告訴 Agent 它是「業務報價前置查詢助理」
#
# Tools：
#   讓 Agent 可以查產品、查成本
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
#
# ✅ 成功畫面：
#   看到「✅ ADK Agent 建立完成」
# ============================================================

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

root_agent = LlmAgent(
    name="quotation_basic_agent",

    model=LiteLlm(
        model=GROQ_MODEL,
        reasoning_effort="none",
    ),

    description="製造業業務報價前置查詢助理",

    instruction="""
你是製造業業務報價助理。

工作原則：
1. 使用者詢問產品能力時，使用 product_lookup。
2. 使用者詢問成本時，使用 latest_cost。
3. 不可自行猜測企業資料。
4. 使用繁體中文回答。
""",

    tools=[
        product_lookup,
        latest_cost
    ],
)

print("✅ ADK Agent 建立完成")


In [ ]:
# ============================================================
# 共用工具｜讓我們可以直接「問 Agent 問題」
# ============================================================
# 這一格的目的：
#   Google ADK 的 Agent 需要 Runner + Session 才能真正執行。
#
# 你可以把它想成：
#
#   Agent = 員工
#   Runner = 讓員工開始工作的執行器
#   Session = 這一次對話的工作區
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import uuid

APP_NAME = "quotation_training"
USER_ID = "student"

async def ask_agent(agent, text: str):
    session_service = InMemorySessionService()
    session_id = str(uuid.uuid4())

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=text)]
    )

    final_answer = "(沒有取得最終回答)"

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                p.text for p in event.content.parts
                if getattr(p, "text", None)
            ]
            if texts:
                final_answer = "\n".join(texts)

    return final_answer

print("✅ Agent 執行工具已準備完成")


## STEP 7｜第一次和 Agent 對話

👨‍🎓 **學生修改區**

你只需要改 `QUESTION` 這一行。


In [ ]:
# ============================================================
# 👨‍🎓 學生修改區
# ============================================================

QUESTION = "CV420H 可以耐幾度？最新成本是多少？"

# ============================================================
# ⚠️ 以下不需要修改
# ============================================================

answer = await ask_agent(root_agent, QUESTION)

print("👤 你的問題：")
print(QUESTION)

print("\n🤖 Agent 回答：")
print(answer)


## ✅ LAB01 你已經完成

你現在已經知道：

- [x] Google ADK 是 Agent Framework
- [x] Groq 提供模型推論
- [x] Tool 是 Agent 可執行的能力
- [x] Agent 可以自己決定要不要使用 Tool
- [x] 企業資料不應該讓 LLM 自己亂猜

### 如果執行失敗

**401 / Unauthorized**  
→ 檢查 Colab Secret：GROQ_API_KEY。

**找不到模型**  
→ 請確認 `GROQ_MODEL` 是否仍可用。

**ModuleNotFoundError**  
→ 重新執行 STEP 1 安裝套件。

下一個 LAB：**把企業 SOP 變成 Agent Skill。**
